# Generic CSV Data Analysis & Group-By Notebook

This notebook is designed to work with **any CSV file** and automatically:

- load and validate the dataset
- standardize common missing-value markers
- profile every column
- identify **categorical, numeric, date/datetime, boolean-like and identifier-like** columns
- optionally handle missing values
- generate overall descriptive statistics
- generate aggregations for categorical dimensions against numeric measures
- evaluate **all categorical combinations** up to a configurable maximum depth
- generate the same aggregations over **Year, Month and ISO Week** for every detected date column
- save profiling and aggregation outputs as CSV files

> **Important:** Generating the mathematical power set of every categorical column can explode exponentially.  
> The notebook therefore defaults to combinations of 1–3 categorical columns. Set `MAX_CATEGORY_COMBINATION`
> higher only when the number/cardinality of categorical fields is manageable.


In [ ]:
# Configuration
from pathlib import Path

# Point this to any CSV file.
CSV_PATH = "/mnt/data/Clarity_Agent_Activation_Sample_Data.csv"

# Output folder
OUTPUT_DIR = Path("/mnt/data/generic_analysis_output")

# CSV settings
CSV_ENCODING = "utf-8"
CSV_SEPARATOR = ","

# Type inference
DATE_PARSE_SUCCESS_THRESHOLD = 0.90
MAX_CATEGORICAL_UNIQUE = 100
MAX_CATEGORICAL_UNIQUE_RATIO = 0.20
ID_UNIQUE_RATIO = 0.95

# Missing-value handling:
# "none"   -> preserve nulls
# "auto"   -> numeric=median, categorical/boolean="Unknown", date unchanged
MISSING_VALUE_STRATEGY = "auto"

# Group-by configuration
MAX_CATEGORY_COMBINATION = 3
MIN_GROUP_ROWS = 1

# Aggregations for every numeric measure
AGGREGATIONS = ["count", "sum", "mean", "median", "min", "max", "std"]

# Safety guard against accidental combinatorial explosion
MAX_GROUPBY_JOBS = 10000

# Optional: manually force classifications if auto-detection needs adjustment
FORCE_DATE_COLUMNS = []
FORCE_CATEGORICAL_COLUMNS = []
FORCE_NUMERIC_COLUMNS = []
EXCLUDE_FROM_GROUPBY = []

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("Output:", OUTPUT_DIR)


In [ ]:
# Imports
import itertools
import math
import re
import warnings
from collections import defaultdict

import numpy as np
import pandas as pd
from IPython.display import display

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 200)


## 1. Load and standardize the CSV


In [ ]:
MISSING_MARKERS = [
    "", " ", "?", "NA", "N/A", "NULL", "null", "None", "none",
    "NaN", "nan", "-", "--", "Unknown?", "#N/A"
]

df_raw = pd.read_csv(
    CSV_PATH,
    sep=CSV_SEPARATOR,
    encoding=CSV_ENCODING,
    low_memory=False,
    na_values=MISSING_MARKERS,
    keep_default_na=True
)

# Clean column names but retain business-readable names.
df_raw.columns = [
    re.sub(r"\\s+", "_", str(c).strip()).strip("_")
    for c in df_raw.columns
]

print(f"Rows: {len(df_raw):,}")
print(f"Columns: {df_raw.shape[1]:,}")
display(df_raw.head())


## 2. Generic data-type inference


In [ ]:
def looks_like_identifier(name, s):
    name_l = name.lower()
    nonnull = s.dropna()
    if len(nonnull) == 0:
        return False

    unique_ratio = nonnull.nunique(dropna=True) / len(nonnull)
    name_hint = bool(re.search(r"(^id$|_id$|^id_|code$|_code$|number$|_no$|name$)", name_l))
    return unique_ratio >= ID_UNIQUE_RATIO and name_hint


def try_numeric(series):
    if pd.api.types.is_numeric_dtype(series):
        return series, 1.0
    s = series.astype("string").str.strip()
    converted = pd.to_numeric(s.str.replace(",", "", regex=False), errors="coerce")
    original_nonnull = s.notna().sum()
    score = converted.notna().sum() / original_nonnull if original_nonnull else 0
    return converted, score


def try_datetime(series, colname):
    if pd.api.types.is_datetime64_any_dtype(series):
        return pd.to_datetime(series, errors="coerce"), 1.0

    s = series.astype("string").str.strip()
    nonnull = s.dropna()
    if len(nonnull) == 0:
        return pd.to_datetime(series, errors="coerce"), 0.0

    # Strong hints reduce false detection of arbitrary text/numbers as dates.
    name_hint = bool(re.search(r"date|time|month|year|week|day|dt$", colname.lower()))
    sample = nonnull.head(min(1000, len(nonnull)))

    # Require a date-like pattern unless the column name itself strongly suggests a date.
    pattern_ratio = sample.str.contains(
        r"(^\\d{4}[-/]\\d{1,2}([-/]\\d{1,2})?$)|"
        r"(^\\d{1,2}[-/]\\d{1,2}[-/]\\d{2,4}$)|"
        r"(^[A-Za-z]{3,9}[ -]\\d{2,4}$)|"
        r"(^\\d{1,2}[ -][A-Za-z]{3,9}[ -]\\d{2,4}$)",
        regex=True, na=False
    ).mean()

    if not name_hint and pattern_ratio < 0.50:
        return pd.to_datetime(series, errors="coerce"), 0.0

    converted = pd.to_datetime(s, errors="coerce")
    score = converted.notna().sum() / s.notna().sum()
    return converted, score


def infer_column_types(df):
    result = {
        "numeric": [],
        "categorical": [],
        "date": [],
        "identifier": [],
        "boolean_like": [],
        "other": []
    }
    converted = df.copy()

    for col in df.columns:
        s = df[col]

        if col in FORCE_DATE_COLUMNS:
            converted[col] = pd.to_datetime(s, errors="coerce")
            result["date"].append(col)
            continue
        if col in FORCE_NUMERIC_COLUMNS:
            converted[col] = pd.to_numeric(s, errors="coerce")
            result["numeric"].append(col)
            continue
        if col in FORCE_CATEGORICAL_COLUMNS:
            converted[col] = s.astype("string")
            result["categorical"].append(col)
            continue

        if looks_like_identifier(col, s):
            result["identifier"].append(col)
            continue

        # Existing numeric dtype wins unless manually overridden.
        if pd.api.types.is_numeric_dtype(s):
            result["numeric"].append(col)
            continue

        dt, dt_score = try_datetime(s, col)
        if dt_score >= DATE_PARSE_SUCCESS_THRESHOLD:
            converted[col] = dt
            result["date"].append(col)
            continue

        num, num_score = try_numeric(s)
        if num_score >= 0.98:
            converted[col] = num
            result["numeric"].append(col)
            continue

        nonnull = s.dropna()
        nunique = nonnull.nunique()
        unique_ratio = nunique / len(nonnull) if len(nonnull) else 0

        normalized = set(nonnull.astype(str).str.strip().str.lower().unique())
        bool_tokens = {"yes", "no", "true", "false", "y", "n", "0", "1"}
        if normalized and normalized.issubset(bool_tokens):
            result["boolean_like"].append(col)
            result["categorical"].append(col)
        elif nunique <= MAX_CATEGORICAL_UNIQUE or unique_ratio <= MAX_CATEGORICAL_UNIQUE_RATIO:
            result["categorical"].append(col)
        else:
            result["other"].append(col)

    return converted, result


df, column_types = infer_column_types(df_raw)

for k, v in column_types.items():
    print(f"\\n{k.upper()} ({len(v)}):")
    print(v)


## 3. Data profiling


In [ ]:
def build_profile(df, types):
    type_lookup = {}
    for typ, cols in types.items():
        for c in cols:
            # categorical is preferred over duplicate boolean_like label
            if c not in type_lookup or typ != "boolean_like":
                type_lookup[c] = typ

    rows = []
    for c in df.columns:
        s = df[c]
        nonnull = s.dropna()
        vc = nonnull.astype(str).value_counts().head(5)

        row = {
            "column": c,
            "inferred_type": type_lookup.get(c, "other"),
            "pandas_dtype": str(s.dtype),
            "row_count": len(s),
            "non_null_count": int(s.notna().sum()),
            "null_count": int(s.isna().sum()),
            "null_pct": round(s.isna().mean() * 100, 4),
            "unique_count": int(nonnull.nunique()),
            "unique_pct_non_null": round(
                (nonnull.nunique() / len(nonnull) * 100) if len(nonnull) else 0, 4
            ),
            "top_values": " | ".join([f"{k}:{v}" for k, v in vc.items()])
        }

        if pd.api.types.is_numeric_dtype(s):
            row.update({
                "min": s.min(),
                "max": s.max(),
                "mean": s.mean(),
                "median": s.median(),
                "std": s.std()
            })
        elif pd.api.types.is_datetime64_any_dtype(s):
            row.update({
                "min": s.min(),
                "max": s.max()
            })

        rows.append(row)

    return pd.DataFrame(rows)

profile_before = build_profile(df, column_types)
display(profile_before)
profile_before.to_csv(OUTPUT_DIR / "01_profile_before_missing_handling.csv", index=False)


## 4. Missing-value handling


In [ ]:
df_clean = df.copy()
imputation_log = []

if MISSING_VALUE_STRATEGY.lower() == "auto":
    for c in column_types["numeric"]:
        if df_clean[c].isna().any():
            fill = df_clean[c].median()
            before = int(df_clean[c].isna().sum())
            df_clean[c] = df_clean[c].fillna(fill)
            imputation_log.append([c, "numeric", before, "median", fill])

    for c in column_types["categorical"]:
        if df_clean[c].isna().any():
            before = int(df_clean[c].isna().sum())
            df_clean[c] = df_clean[c].astype("string").fillna("Unknown")
            imputation_log.append([c, "categorical", before, "Unknown", "Unknown"])

    # Dates are deliberately not fabricated. Missing dates remain NaT.

imputation_log = pd.DataFrame(
    imputation_log,
    columns=["column", "type", "missing_before", "method", "fill_value"]
)

display(imputation_log if len(imputation_log) else pd.DataFrame({"message":["No imputations required"]}))
imputation_log.to_csv(OUTPUT_DIR / "02_imputation_log.csv", index=False)

profile_after = build_profile(df_clean, column_types)
profile_after.to_csv(OUTPUT_DIR / "03_profile_after_missing_handling.csv", index=False)


## 5. Numeric descriptive statistics


In [ ]:
numeric_cols = [c for c in column_types["numeric"] if c not in EXCLUDE_FROM_GROUPBY]
categorical_cols = [
    c for c in column_types["categorical"]
    if c not in EXCLUDE_FROM_GROUPBY and c not in column_types["identifier"]
]
date_cols = [c for c in column_types["date"] if c not in EXCLUDE_FROM_GROUPBY]

numeric_summary = df_clean[numeric_cols].describe(
    percentiles=[.01,.05,.10,.25,.50,.75,.90,.95,.99]
).T if numeric_cols else pd.DataFrame()

display(numeric_summary)
numeric_summary.to_csv(OUTPUT_DIR / "04_numeric_summary.csv")


## 6. Build categorical combinations

The notebook produces every combination from one categorical dimension through
`MAX_CATEGORY_COMBINATION`. For example, with `Zone`, `State`, `Channel`:

- Zone
- State
- Channel
- Zone + State
- Zone + Channel
- State + Channel
- Zone + State + Channel

Identifier-like fields are intentionally excluded from this step.


In [ ]:
def all_dimension_combinations(columns, max_depth):
    combos = []
    max_depth = min(max_depth, len(columns))
    for r in range(1, max_depth + 1):
        combos.extend(itertools.combinations(columns, r))
    return combos

dimension_combos = all_dimension_combinations(
    categorical_cols,
    MAX_CATEGORY_COMBINATION
)

estimated_jobs = len(dimension_combos) * max(1, len(numeric_cols))
print(f"Categorical columns used: {len(categorical_cols)}")
print(f"Dimension combinations: {len(dimension_combos):,}")
print(f"Numeric measures: {len(numeric_cols)}")
print(f"Approx. overall measure jobs: {estimated_jobs:,}")

if estimated_jobs > MAX_GROUPBY_JOBS:
    raise ValueError(
        f"Safety limit exceeded: {estimated_jobs:,} jobs > {MAX_GROUPBY_JOBS:,}. "
        "Reduce MAX_CATEGORY_COMBINATION, exclude high-cardinality dimensions, "
        "or increase MAX_GROUPBY_JOBS intentionally."
    )


## 7. Overall group-by analysis


In [ ]:
def safe_filename(text, max_len=180):
    text = re.sub(r"[^A-Za-z0-9_.-]+", "_", text)
    return text[:max_len]


def aggregate_measure(data, dimensions, measure):
    temp = data[list(dimensions) + [measure]].copy()

    out = (
        temp.groupby(list(dimensions), dropna=False, observed=True)[measure]
        .agg(AGGREGATIONS)
        .reset_index()
    )

    # Add group row count independently of numeric null behavior.
    rows = (
        temp.groupby(list(dimensions), dropna=False, observed=True)
        .size()
        .rename("group_rows")
        .reset_index()
    )
    out = rows.merge(out, on=list(dimensions), how="left")
    out.insert(len(dimensions), "measure", measure)
    out.insert(len(dimensions) + 1, "dimension_set", " + ".join(dimensions))
    return out[out["group_rows"] >= MIN_GROUP_ROWS]


overall_manifest = []
overall_dir = OUTPUT_DIR / "overall_groupbys"
overall_dir.mkdir(exist_ok=True)

for dims in dimension_combos:
    for measure in numeric_cols:
        result = aggregate_measure(df_clean, dims, measure)
        filename = safe_filename(
            f"{'__'.join(dims)}___{measure}.csv"
        )
        result.to_csv(overall_dir / filename, index=False)
        overall_manifest.append({
            "period": "overall",
            "date_column": None,
            "dimensions": " + ".join(dims),
            "measure": measure,
            "rows_generated": len(result),
            "file": str(overall_dir / filename)
        })

overall_manifest = pd.DataFrame(overall_manifest)
overall_manifest.to_csv(OUTPUT_DIR / "05_overall_groupby_manifest.csv", index=False)
display(overall_manifest.head(20))
print(f"Overall group-by files generated: {len(overall_manifest):,}")


## 8. Date expansion: Year, Month and ISO Week


In [ ]:
def add_date_grains(data, date_col):
    x = data.copy()
    dt = pd.to_datetime(x[date_col], errors="coerce")

    prefix = safe_filename(date_col)
    x[f"{prefix}__Year"] = dt.dt.year.astype("Int64")
    x[f"{prefix}__Month"] = dt.dt.to_period("M").astype("string")

    iso = dt.dt.isocalendar()
    x[f"{prefix}__ISO_Year"] = iso.year.astype("Int64")
    x[f"{prefix}__ISO_Week"] = iso.week.astype("Int64")
    x[f"{prefix}__YearWeek"] = (
        x[f"{prefix}__ISO_Year"].astype("string")
        + "-W"
        + x[f"{prefix}__ISO_Week"].astype("string").str.zfill(2)
    )

    return x, {
        "year": f"{prefix}__Year",
        "month": f"{prefix}__Month",
        "week": f"{prefix}__YearWeek"
    }

for d in date_cols:
    print(d, "->", df_clean[d].min(), "to", df_clean[d].max())


## 9. Time-spread group-by analysis


In [ ]:
time_manifest = []
time_dir = OUTPUT_DIR / "time_groupbys"
time_dir.mkdir(exist_ok=True)

for date_col in date_cols:
    dated, grains = add_date_grains(df_clean, date_col)

    for grain_name, grain_col in grains.items():
        grain_dir = time_dir / safe_filename(date_col) / grain_name
        grain_dir.mkdir(parents=True, exist_ok=True)

        for dims in dimension_combos:
            dimensions = (grain_col,) + tuple(dims)

            for measure in numeric_cols:
                result = aggregate_measure(dated, dimensions, measure)

                filename = safe_filename(
                    f"{grain_name}__{'__'.join(dims)}___{measure}.csv"
                )
                path = grain_dir / filename
                result.to_csv(path, index=False)

                time_manifest.append({
                    "period": grain_name,
                    "date_column": date_col,
                    "dimensions": " + ".join(dims),
                    "measure": measure,
                    "rows_generated": len(result),
                    "file": str(path)
                })

time_manifest = pd.DataFrame(time_manifest)
time_manifest.to_csv(OUTPUT_DIR / "06_time_groupby_manifest.csv", index=False)
display(time_manifest.head(20))
print(f"Time group-by files generated: {len(time_manifest):,}")


## 10. Optional consolidated long-format analytical table


In [ ]:
# This produces one searchable long-format table from all generated group-by CSVs.
# Disable or sample it for very large/high-cardinality datasets.

BUILD_CONSOLIDATED_OUTPUT = True

if BUILD_CONSOLIDATED_OUTPUT:
    consolidated_parts = []

    manifests = [overall_manifest]
    if len(time_manifest):
        manifests.append(time_manifest)

    manifest_all = pd.concat(manifests, ignore_index=True)

    for _, r in manifest_all.iterrows():
        part = pd.read_csv(r["file"])
        part["analysis_period"] = r["period"]
        part["analysis_date_column"] = r["date_column"]
        consolidated_parts.append(part)

    if consolidated_parts:
        consolidated = pd.concat(consolidated_parts, ignore_index=True, sort=False)
        consolidated.to_csv(
            OUTPUT_DIR / "07_consolidated_groupby_analysis.csv",
            index=False
        )
        print(f"Consolidated rows: {len(consolidated):,}")
        display(consolidated.head(30))


## 11. Final validation and output inventory


In [ ]:
inventory = []

for p in sorted(OUTPUT_DIR.rglob("*.csv")):
    inventory.append({
        "file": str(p.relative_to(OUTPUT_DIR)),
        "size_mb": round(p.stat().st_size / 1024 / 1024, 3)
    })

inventory = pd.DataFrame(inventory)
display(inventory.head(50))
print(f"Total CSV outputs: {len(inventory):,}")
print(f"Total output size: {inventory['size_mb'].sum():,.2f} MB" if len(inventory) else "No outputs")
print("\nAnalysis complete.")


## Scaling notes

For a small/medium CSV, the notebook can run as written. For large enterprise datasets:

1. Keep `MAX_CATEGORY_COMBINATION` at 1–3.
2. Exclude identifiers, free-text fields and very high-cardinality dimensions.
3. Select only business-relevant numeric measures if there are hundreds of numeric columns.
4. Avoid building the consolidated output when the individual group-by files are already sufficient.
5. For datasets that no longer fit comfortably in memory, retain this analytical design but implement the execution layer with **PySpark / Databricks** rather than pandas.
